# 02 — Exploratory Data Analysis (EDA)
**AI-Driven Waste Collection & Route Optimization**

Analyse waste generation patterns, correlations, and seasonal effects.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sensor_df = pd.read_csv('../data/sensor_readings.csv', parse_dates=['timestamp'])
bins_df   = pd.read_csv('../data/bins_metadata.csv')
sensor_df = sensor_df.merge(bins_df[['bin_id','x_coord','y_coord','fill_rate_pct_per_hr']], on='bin_id')

sensor_df['hour']    = sensor_df['timestamp'].dt.hour
sensor_df['dow']     = sensor_df['timestamp'].dt.dayofweek
sensor_df['dow_name']= sensor_df['timestamp'].dt.day_name()
sensor_df['date']    = sensor_df['timestamp'].dt.date
sensor_df['week']    = sensor_df['timestamp'].dt.isocalendar().week
print(f'Loaded {len(sensor_df):,} records across {sensor_df.bin_id.nunique()} bins.')

## 1. Overflow Risk Analysis

In [ ]:
overflow_threshold = 80
overflow_events = sensor_df[sensor_df['fill_level_pct'] >= overflow_threshold]
overflow_rate = len(overflow_events) / len(sensor_df) * 100

overflow_by_zone = sensor_df.groupby('zone_type').apply(
    lambda x: (x['fill_level_pct'] >= overflow_threshold).sum() / len(x) * 100
).reset_index()
overflow_by_zone.columns = ['zone_type', 'overflow_risk_pct']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Overflow Risk & Temporal Patterns', fontsize=13, fontweight='bold')

# Overflow by zone
axes[0].bar(overflow_by_zone['zone_type'], overflow_by_zone['overflow_risk_pct'],
            color=['#FF6B6B','#FFA07A','#FFD700','#90EE90'])
axes[0].set_title(f'Overflow Risk by Zone\n(threshold >= {overflow_threshold}%)')
axes[0].set_ylabel('% readings at overflow risk')
axes[0].grid(True, alpha=0.3, axis='y')

# Heatmap: hour x day
pivot = sensor_df.groupby(['dow', 'hour'])['fill_level_pct'].mean().unstack()
days  = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
pivot.index = days
sns.heatmap(pivot, ax=axes[1], cmap='RdYlGn_r', cbar_kws={'label':'Avg Fill %'})
axes[1].set_title('Avg Fill Level Heatmap\n(Day vs Hour)')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Day of Week')

# Daily avg fill trend
daily_avg = sensor_df.groupby('date')['fill_level_pct'].mean()
axes[2].plot(list(daily_avg.index), daily_avg.values, color='#2196F3', linewidth=1.5)
axes[2].fill_between(range(len(daily_avg)), daily_avg.values, alpha=0.2, color='#2196F3')
axes[2].set_title('Daily Average Fill Level Trend')
axes[2].set_xlabel('Day')
axes[2].set_ylabel('Avg Fill Level (%)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/02_eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Global overflow risk: {overflow_rate:.1f}%')

## 2. Correlation & Statistical Analysis

In [ ]:
# Bin-level summary stats
bin_stats = sensor_df.groupby('bin_id').agg(
    avg_fill      = ('fill_level_pct','mean'),
    max_fill      = ('fill_level_pct','max'),
    std_fill      = ('fill_level_pct','std'),
    fill_rate     = ('fill_rate_pct_per_hr','first'),
    zone_type     = ('zone_type','first'),
).reset_index()

r, p = stats.pearsonr(bin_stats['fill_rate'], bin_stats['avg_fill'])
print(f'Pearson r(fill_rate, avg_fill) = {r:.3f}  (p={p:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Statistical Correlation Analysis', fontsize=13, fontweight='bold')

colors = {'residential':'#4CAF50','commercial':'#FF9800','industrial':'#F44336','park':'#2196F3'}
for zone in bin_stats['zone_type'].unique():
    sub = bin_stats[bin_stats['zone_type'] == zone]
    axes[0].scatter(sub['fill_rate'], sub['avg_fill'], label=zone,
                    color=colors.get(zone,'grey'), s=60, alpha=0.8)

m, b = np.polyfit(bin_stats['fill_rate'], bin_stats['avg_fill'], 1)
x_line = np.linspace(bin_stats['fill_rate'].min(), bin_stats['fill_rate'].max(), 100)
axes[0].plot(x_line, m*x_line+b, 'k--', linewidth=1.5, label=f'Trend (r={r:.2f})')
axes[0].set_xlabel('Fill Rate (%/hr)')
axes[0].set_ylabel('Avg Fill Level (%)')
axes[0].set_title('Fill Rate vs Avg Fill Level')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

zone_order = ['residential','commercial','industrial','park']
data_to_box = [bin_stats[bin_stats['zone_type']==z]['avg_fill'].values for z in zone_order]
bp = axes[1].boxplot(data_to_box, labels=zone_order, patch_artist=True)
for patch, zone in zip(bp['boxes'], zone_order):
    patch.set_facecolor(colors.get(zone,'grey'))
    patch.set_alpha(0.7)
axes[1].set_title('Fill Level Distribution by Zone')
axes[1].set_ylabel('Avg Fill Level (%)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/02_eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
bin_stats.groupby('zone_type')[['avg_fill','max_fill','std_fill']].mean().round(2)